# A3b: Architecture Comparison on Leakage-Feedback Data

This trains the same four operator-learning architectures as the throttled-data comparison
(`kaggle_a3_throttled_arch_comparison.ipynb`), on the geometry1 leakage-feedback pilot
(`goal.md` Track D, `docs/report.md` §9.10). This is the sharpest nonlinearity test in this
benchmark so far, since leakage is positive feedback (hotter leads to more leakage power,
which makes it hotter still) rather than throttling's self-limiting negative feedback.

| architecture | conditioning | physics loss |
|---|---|---|
| `fno` | none (baseline) | off |
| `cond-fno` | FiLM on HTC/T_amb/TSV_frac | off |
| `cno-fno` | FiLM + CNO multiscale | on (PDE + flux) |
| `cno-fno` + attention | FiLM + CNO + axial self-attention (SAU-FNO style) | on (PDE + flux) |

Why this matters: on the throttled pilot, no CPU-scale FNO run has beaten ridge (det.MAE
1.795 K vs ridge's 0.707 K). Leakage feedback is structurally harder: 30% of the
20-scenario pilot (6/20) ran into thermal runaway with no steady state at all, and among
the 13 that converged, ridge still won (det.MAE 0.248 K, spatial R² 0.972, matching the
un-throttled baseline). No FNO of any capacity has been run on this data yet, so this
notebook is that first run, at GPU scale rather than another CPU smoke test.

Critical: this notebook trains only on the converged subset. Runaway scenarios have no
physically meaningful steady-state temperature field (some diverge past 500,000°C, a
numerical artifact of the fixed-point iteration hitting its cap, not a real material
state). Training or scoring against them would test whether a network can fit numerical
garbage, not whether it models the physics. The filtering below is automatic (reads
`leakage_converged` / `leakage_runaway` from each file's metadata); do not disable it.

The bar to beat (already measured on the CPU, `scripts/baselines_leakage.py`,
converged-only split, 9 train / 4 test):

| baseline | det.MAE (K) | spatial R² | hotspot loc. err (µm) |
|---|---|---|---|
| ridge (nominal power) | 0.248 | 0.972 | 0 |
| kNN (k=3, nominal power) | 0.297 | 0.965 | 0 |
| nearest-neighbour | 0.476 | 0.919 | 0 |
| mean | 0.716 | 0.754 | 1803 |

Reference context from the throttled comparison (different mechanism, same benchmark):
ridge scored det.MAE 0.707 K / R² 0.919 there, meaning ridge does better on convergent
leakage data than on throttled data despite leakage being the more severe nonlinearity in
principle. That asymmetry (a harder mechanism, but only when it doesn't diverge, produces
an easier-to-fit dataset) is worth reporting regardless of what this notebook finds.

If none of the four architectures beat ridge here either, that's a second, independent
data point for the same conclusion the throttled comparison is testing. If one does,
report it: this would be the first regime in the whole benchmark where a neural operator
wins.

## Kaggle Dataset Setup

1. Source code: upload `src/` (reuse the `thermo-pinn-src` slug from the throttled-data
   notebook if you already have it).
2. Leakage pilot data: zip and upload the contents of
   `data/3d-ice-leakage-pilot/geometry1/` (20 `.npz` files, of which this notebook will use
   only the 13 that converged, see filtering below). Slug suggestion:
   `3dice-leakage-pilot-data`.
3. Enable GPU accelerator (Settings -> Accelerator -> GPU T4 x1).
4. Run all cells.

Expected time: smaller converged split than the throttled comparison (9 train vs 15), so
expect similar or slightly faster per-model time, roughly 3-8 min/model at 400 epochs.


In [ ]:
import subprocess, sys

# Install any missing packages (PyYAML is the only non-standard dep)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)

import os
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: no GPU detected -- enable it under Settings -> Accelerator before running the training cells.')


In [ ]:
import sys
from pathlib import Path

# -- Kaggle dataset paths ----------------------------------------------------
# Adjust these slugs to match the datasets you attached to this notebook.
SRC_DATASET  = 'thermo-pinn-src'          # dataset containing the src/ folder
DATA_DATASET = '3dice-leakage-pilot-data'  # dataset containing the leakage geometry1 .npz files

SRC_ROOT  = Path(f'/kaggle/input/{SRC_DATASET}')
DATA_ROOT = Path(f'/kaggle/input/{DATA_DATASET}')
OUT_DIR   = Path('/kaggle/working/checkpoints/a3b_leakage_arch_comparison')

sys.path.insert(0, str(SRC_ROOT))

assert SRC_ROOT.exists(),  f'Source dataset not found at {SRC_ROOT}. Check SRC_DATASET slug.'
assert DATA_ROOT.exists(), f'Data dataset not found at {DATA_ROOT}. Check DATA_DATASET slug.'
print('Source root:', SRC_ROOT)
print('Data root:  ', DATA_ROOT)


In [ ]:
import logging, time, json
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.core.geometry_builders import get_geometry_by_name
from src.pinn.data_loader import NormStats, compute_norm_stats
from src.fno.model import build_fno, build_cond_fno, build_cno_fno
from src.fno.data_loader import FNODataset, predict_to_flat
from src.fno.trainer import FNOTrainer

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)-8s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
print('Imports OK')


In [ ]:
def metrics(pred: np.ndarray, true: np.ndarray, coords: np.ndarray) -> dict:
    """
    Same detrended-metric definition as scripts/baselines.py::metrics (not
    importable here since only src/ is uploaded to Kaggle) -- kept byte-for-byte
    identical so results are directly comparable to the ridge/kNN numbers in
    docs/report.md.
    """
    err = pred - true
    ss_res = float(np.sum(err ** 2))
    ss_tot = float(np.sum((true - true.mean()) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')

    d_pred, d_true = pred - pred.mean(), true - true.mean()
    d_err = d_pred - d_true
    d_ss_res = float(np.sum(d_err ** 2))
    d_ss_tot = float(np.sum(d_true ** 2))
    spatial_r2 = 1.0 - d_ss_res / d_ss_tot if d_ss_tot > 0 else float('nan')

    hot_true_idx = int(np.argmax(true))
    hot_pred_idx = int(np.argmax(pred))
    hotspot_loc_err_um = float(np.linalg.norm(coords[hot_true_idx] - coords[hot_pred_idx]))
    hotspot_temp_err_K = float(pred[hot_true_idx] - true[hot_true_idx])

    return {
        'mae_K': float(np.mean(np.abs(err))),
        'rmse_K': float(np.sqrt(np.mean(err ** 2))),
        'max_abs_err_K': float(np.max(np.abs(err))),
        'r2': r2,
        'mae_detrended_K': float(np.mean(np.abs(d_err))),
        'spatial_r2': spatial_r2,
        'true_spatial_std_K': float(true.std()),
        'hotspot_temp_err_K': hotspot_temp_err_K,
        'hotspot_loc_err_um': hotspot_loc_err_um,
    }

print('metrics() defined')


In [ ]:
GEOM_NAME    = 'geometry1'
CHANNELS     = 32
N_BLOCKS     = 4
N_HEADS      = 4             # CNO-FNO axial attention heads; must divide CHANNELS evenly
MODES        = (16, 16, 12)  # fno/cond-fno spectral mode counts
EPOCHS       = 400           # same budget for all 4 models -- fair comparison, not each model's own default
BATCH_SIZE   = 4
LR           = 1e-3
PDE_WEIGHT   = 0.1           # only applied to the two cno-fno variants (physics=on)
FLUX_WEIGHT  = 0.5           # interface-isolated flux-continuity loss, cno-fno variants only

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')
print(f'Config: channels={CHANNELS} blocks={N_BLOCKS} epochs={EPOCHS}')


In [ ]:
geometry = get_geometry_by_name(GEOM_NAME)
geometries = {GEOM_NAME: geometry}
grid_shape = geometry.mesh_resolution
print(geometry.summary())
print('Grid shape (declared):', grid_shape)


In [ ]:
# Load and filter to CONVERGED scenarios only -- see the markdown intro for why this
# is not optional. A file is used only if leakage_converged=True AND leakage_runaway=False;
# everything else (runaway, or "neither" -- still climbing at the iteration cap) is
# excluded before the train/test split is even formed, mirroring
# scripts/baselines_leakage.py's filtering exactly so the CPU reference numbers above are
# a fair comparison.
all_files = sorted(DATA_ROOT.rglob(f'{GEOM_NAME}_*.npz'))
if not all_files:
    all_files = sorted(DATA_ROOT.glob(f'{GEOM_NAME}_*.npz'))
assert all_files, f'No files found in {DATA_ROOT}. Check DATA_DATASET slug and file naming.'

converged_files, runaway_files, neither_files = [], [], []
for f in all_files:
    d = np.load(f, allow_pickle=True)
    meta = dict(d['metadata'][0])
    assert meta.get('leakage_enabled'), (
        f'{f.name} has no leakage_enabled metadata -- check DATA_DATASET points at the '
        'leakage pilot, not the standard data/3d-ice/ dataset.'
    )
    if meta.get('leakage_runaway'):
        runaway_files.append(f)
    elif meta.get('leakage_converged'):
        converged_files.append(f)
    else:
        neither_files.append(f)

print(f'{len(all_files)} total: {len(converged_files)} converged, '
      f'{len(runaway_files)} runaway (excluded), {len(neither_files)} neither (excluded)')

train_files = [f for f in converged_files if '_train_' in f.stem]
test_files  = [f for f in converged_files if '_test_' in f.stem]
print(f'Converged split -> train: {len(train_files)}  test: {len(test_files)}')
assert len(train_files) >= 3 and len(test_files) >= 2, (
    'Too few converged scenarios for a meaningful fit -- this pilot batch may need to be '
    'regenerated with gentler leakage settings before running this notebook.'
)

_probe = np.load(train_files[0], allow_pickle=True)
assert 'power_nominal' in _probe.files, (
    'power_nominal missing from this .npz -- re-export with the current '
    'src/export/npz_exporter.py before uploading.'
)


In [ ]:
# Take the z-depth from the DATA, not geometry.mesh_resolution -- 3D-ICE emits one
# value per stack element (sub-layer discretisation), not the declared mesh nz.
# Mirrors scripts/train_fno.py's grid-from-data logic.
_d = np.load(train_files[0], allow_pickle=True)
_n = _d['coords'].shape[0]
_nx, _ny = grid_shape[0], grid_shape[1]
if _nx * _ny > 0 and _n % (_nx * _ny) == 0:
    data_grid = (_nx, _ny, _n // (_nx * _ny))
    if data_grid != tuple(grid_shape):
        print(f'Grid from data: {data_grid} (geometry declares {tuple(grid_shape)})')
        grid_shape = data_grid
print('Using grid_shape:', grid_shape)


In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
norm_path = OUT_DIR / 'norm_stats.json'

if norm_path.exists():
    norm_stats = NormStats.load(norm_path)
    print('Loaded existing norm stats from', norm_path)
else:
    print('Computing norm stats over', len(train_files), 'training files...')
    norm_stats = compute_norm_stats(train_files, geometries)
    norm_stats.save(norm_path)
    print('Saved norm stats to', norm_path)

print(f'T range: [{norm_stats.T_min:.1f}, {norm_stats.T_max:.1f}] K')


In [ ]:
print('Loading training dataset (FNODataset builds Q_norm from power_nominal automatically'
      ' when the field is present -- the leakage-delivered power is amplified relative to'
      ' the request, so this asks the same "requested power -> resolved temperature"'
      ' question ridge is scored on, not the easier already-resolved-power problem)...')
train_dataset = FNODataset(train_files, norm_stats, grid_shape)
val_dataset   = FNODataset(test_files, norm_stats, grid_shape)

print(f'Train: {len(train_dataset)} scenarios')
print(f'Val  : {len(val_dataset)} scenarios')


In [ ]:
models = {
    'fno': build_fno(
        grid_shape=grid_shape, modes=MODES, hidden_ch=CHANNELS, n_blocks=N_BLOCKS, device=DEVICE,
    ),
    'cond-fno': build_cond_fno(
        grid_shape=grid_shape, modes=MODES, hidden_ch=CHANNELS, n_blocks=N_BLOCKS, device=DEVICE,
    ),
    'cno-fno': build_cno_fno(
        grid_shape=grid_shape, ch=CHANNELS, n_fno_blocks=N_BLOCKS,
        use_attention=False, device=DEVICE,
    ),
    'cno-fno+attn (SAU)': build_cno_fno(
        grid_shape=grid_shape, ch=CHANNELS, n_fno_blocks=N_BLOCKS,
        use_attention=True, n_heads=N_HEADS, device=DEVICE,
    ),
}

for name, m in models.items():
    print(f'{name:<22} {m.n_parameters:>12,} params')


In [ ]:
# physics loss only applies to the two cno-fno variants (matches _MODEL_DEFAULTS# in scripts/train_fno.py: physics on for cno-fno, off for fno/cond-fno)physics_on = {'fno': False, 'cond-fno': False, 'cno-fno': True, 'cno-fno+attn (SAU)': True}trainers = {}times = {}for name, model in models.items():    pde_w  = PDE_WEIGHT  if physics_on[name] else 0.0    flux_w = FLUX_WEIGHT if physics_on[name] else 0.0    out_dir = OUT_DIR / name.replace(' ', '_').replace('(', '').replace(')', '')    print(f'\n=== Training {name} (physics={physics_on[name]}) ===')    t0 = time.time()    trainer = FNOTrainer(        model=model, norm_stats=norm_stats, train_data=train_dataset, val_data=val_dataset,        output_dir=out_dir, batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, device=DEVICE,        geometry_name=f'{GEOM_NAME}_{name}', pde_weight=pde_w, flux_weight=flux_w,        geometry=geometry, log_interval=max(1, EPOCHS // 10),        use_amp=False,  # cuFFT half-precision needs power-of-two grids; ours never are    )    trainer.train()    elapsed = time.time() - t0    trainers[name] = trainer    times[name] = elapsed    print(f'{name}: best val MAE={trainer.best_val_mae:.3f} K, time={elapsed/60:.1f} min')

In [ ]:
# Evaluate every model on the test set with the SAME detrended metrics ridge/kNN
# were scored on (scripts/baselines_leakage.py), for a direct, apples-to-apples comparison.
nx, ny, nz = grid_shape
xs = np.linspace(0, geometry.die_width, nx)
ys = np.linspace(0, geometry.die_length, ny)
zs = np.linspace(0, geometry.get_total_height(), nz)
X, Y, Z = np.meshgrid(xs, ys, zs, indexing='ij')
coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)

results = {}
for name, model in models.items():
    model.eval()
    all_m = []
    with torch.no_grad():
        for item in val_dataset.items:
            T_pred_K, T_true_K = predict_to_flat(model, item, DEVICE, norm_stats)
            m = metrics(T_pred_K, T_true_K, coords)
            m['name'] = item['name']
            all_m.append(m)
    agg = {k: float(np.mean([m[k] for m in all_m]))
           for k in ('mae_K', 'mae_detrended_K', 'spatial_r2', 'hotspot_loc_err_um')}
    results[name] = agg
    print(name, agg)


In [ ]:
REFERENCE = {
    'ridge (nominal power)':      {'mae_detrended_K': 0.248, 'spatial_r2': 0.972, 'hotspot_loc_err_um': 0.0},
    'kNN, k=3 (nominal power)':   {'mae_detrended_K': 0.297, 'spatial_r2': 0.965, 'hotspot_loc_err_um': 0.0},
    'nearest-neighbour':         {'mae_detrended_K': 0.476, 'spatial_r2': 0.919, 'hotspot_loc_err_um': 0.0},
    'mean':                      {'mae_detrended_K': 0.716, 'spatial_r2': 0.754, 'hotspot_loc_err_um': 1803.1},
}

print(f'{"model":<24} {"det.MAE (K)":>12} {"spatial R2":>12} {"hotspot err (um)":>18}')
print('-' * 70)
for name, r in REFERENCE.items():
    hot = f'{r["hotspot_loc_err_um"]:.0f}' if r['hotspot_loc_err_um'] is not None else '--'
    print(f'{name:<24} {r["mae_detrended_K"]:>12.3f} {r["spatial_r2"]:>12.3f} {hot:>18}')
print('-' * 70)
for name, r in results.items():
    beat_ridge = r['spatial_r2'] > REFERENCE['ridge (nominal power)']['spatial_r2']
    marker = '  <-- BEATS RIDGE' if beat_ridge else ''
    print(f'{name:<24} {r["mae_detrended_K"]:>12.3f} {r["spatial_r2"]:>12.3f} {r["hotspot_loc_err_um"]:>18.0f}{marker}')


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = {'fno': 'gray', 'cond-fno': 'steelblue', 'cno-fno': 'darkorange', 'cno-fno+attn (SAU)': 'crimson'}
for name, trainer in trainers.items():
    ax.plot(trainer.history['epoch'], trainer.history['val_mae_K'], label=name, color=colors.get(name))
ax.axhline(REFERENCE['ridge (nominal power)']['mae_detrended_K'], color='black', linestyle='--',
           label='ridge det.MAE (reference, not directly comparable to raw val MAE)', alpha=0.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('Val MAE (K)')
ax.set_title(f'{GEOM_NAME} (leakage pilot, converged-only): architecture comparison')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
summary = {
    'geometry': GEOM_NAME,
    'grid_shape': list(grid_shape),
    'epochs': EPOCHS,
    'channels': CHANNELS,
    'n_blocks': N_BLOCKS,
    'n_converged': len(converged_files),
    'n_runaway_excluded': len(runaway_files),
    'reference': REFERENCE,
    'results': results,
    'params': {name: m.n_parameters for name, m in models.items()},
    'train_time_min': {name: t / 60 for name, t in times.items()},
}

with open(OUT_DIR / 'a3b_leakage_arch_comparison_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print()
print('Download this JSON plus the *_best.pt checkpoints from /kaggle/working/checkpoints/')
print('and report the results table back -- goal.md Track D / docs/report.md Sec 9.10')
print('are the two places this result needs to be recorded.')
